# 02 — Modeling Walkthrough
## Bank Customer Churn — European Dataset

This notebook walks through the complete modeling pipeline:
1. Data preparation + feature engineering
2. All 5 model families with tuning
3. Evaluation and comparison
4. SHAP explainability
5. Final scoring

In [ ]:
import os, sys
sys.path.insert(0, os.path.join('..', 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, joblib
warnings.filterwarnings('ignore')

from preprocessing import load_data, TARGET, NUMERIC_FEATURES, CATEGORICAL_FEATURES
from feature_engineering import add_engineered_features
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
MODELS_DIR = os.path.join('..', 'models')
DATA_PROCESSED = os.path.join('..', 'data', 'processed')
FIGURES_DIR = os.path.join('..', 'reports', 'figures')

# Load and engineer features
df = load_data()
print(f'Raw data: {df.shape}')
df = add_engineered_features(df)
print(f'After engineering: {df.shape}')
print(f'Churn rate: {df[TARGET].mean():.2%}')

feature_cols = [c for c in df.columns if c != TARGET and c != 'Age_Band']
X = df[feature_cols]
y = df[TARGET]
print(f'\nFeatures: {len(feature_cols)}')
print(feature_cols)

## 1. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print('Train/test split (80/20 stratified):')
print(f'  Train: {len(X_train):,} samples | Churn rate: {y_train.mean():.2%}')
print(f'  Test:  {len(X_test):,} samples  | Churn rate: {y_test.mean():.2%}')
print('\nStratification confirmed: churn rate is preserved in both splits.')

## 2. Feature Engineering Impact Analysis

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from preprocessing import build_preprocessor

base_numeric = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary']
engineered = ['Balance_to_Salary', 'Products_per_Tenure', 'Active_Products',
              'Age_Tenure_Interaction', 'Tenure_Age_Ratio', 'Zero_Balance_Flag']

def quick_rf_auc(X_tr, y_tr, X_te, y_te, numeric_feats):
    pre = build_preprocessor(numeric_features=numeric_feats)
    pipe = Pipeline([('pre', pre), ('clf', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))])
    pipe.fit(X_tr, y_tr)
    return roc_auc_score(y_te, pipe.predict_proba(X_te)[:, 1])

baseline_auc = quick_rf_auc(X_train, y_train, X_test, y_test, base_numeric)
print(f'Baseline AUC (no engineered features): {baseline_auc:.4f}')

rows = []
for feat in engineered:
    feats = [f for f in base_numeric + [feat] if f in X_train.columns]
    auc = quick_rf_auc(X_train, y_train, X_test, y_test, feats)
    rows.append({'Feature': feat, 'AUC': round(auc, 5), 'Delta': round(auc - baseline_auc, 5), 'Improved': auc > baseline_auc})
    print(f'  + {feat}: AUC={auc:.4f} ({"▲" if auc > baseline_auc else "▼"}{auc-baseline_auc:+.4f})')

fe_df = pd.DataFrame(rows).sort_values('Delta', ascending=False)
print(f'\nAll-features AUC: {quick_rf_auc(X_train, y_train, X_test, y_test, [f for f in base_numeric + engineered if f in X_train.columns]):.4f}')

## 3. Load Pre-trained Models (from src/train.py)

In [ ]:
import json

# Load pre-trained pipelines (run src/train.py first)
all_pipelines = joblib.load(os.path.join(MODELS_DIR, 'all_pipelines.joblib'))
best_pipeline = joblib.load(os.path.join(MODELS_DIR, 'best_model_pipeline.joblib'))

with open(os.path.join(MODELS_DIR, 'training_metadata.json')) as f:
    metadata = json.load(f)

print(f'Best model: {metadata["best_model"]}')
print(f'Optimal threshold: {metadata["optimal_threshold"]:.2f}')
print(f'Models available: {list(all_pipelines.keys())}')

## 4. Full Metrics Comparison

In [ ]:
metrics_df = pd.read_csv(os.path.join(DATA_PROCESSED, 'model_comparison_full.csv'))
print('Model Comparison Table:')
display_cols = ['Model', 'Accuracy', 'Precision', 'Recall', 'F1_Score', 'ROC_AUC_Test', 'ROC_AUC_CV_Mean', 'ROC_AUC_CV_Std']
display_cols = [c for c in display_cols if c in metrics_df.columns]
metrics_df[display_cols].style.highlight_max(subset=['ROC_AUC_Test', 'F1_Score', 'Recall'], color='#D1FAE5').format(
    {c: '{:.4f}' for c in display_cols if c != 'Model'}
)

## 5. ROC Curves

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

fig, ax = plt.subplots(figsize=(9, 7))
ax.plot([0, 1], [0, 1], 'k--', label='Random')

PALETTE = ['#2563EB', '#10B981', '#F59E0B', '#EF4444', '#8B5CF6']
for (name, pipe), color in zip(all_pipelines.items(), PALETTE):
    y_prob = pipe.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, lw=2.5, color=color, label=f'{name} (AUC={auc:.4f})')

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — All Models', fontsize=14, fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 6. SHAP Explainability

In [ ]:
import shap
from preprocessing import get_feature_names

# Use best pipeline
preprocessor = best_pipeline.named_steps['preprocessor']
classifier = best_pipeline.named_steps['classifier']

numeric_features = [c for c in NUMERIC_FEATURES if c in X_test.columns]
feature_names = get_feature_names(preprocessor, numeric_features, CATEGORICAL_FEATURES)

X_sample = X_test.sample(500, random_state=42)
X_transformed = preprocessor.transform(X_sample)
X_df = pd.DataFrame(X_transformed, columns=feature_names)

explainer = shap.TreeExplainer(classifier)
shap_vals = explainer.shap_values(X_df)
if isinstance(shap_vals, list):
    shap_vals = shap_vals[1]

print(f'SHAP values computed for {len(X_sample)} samples, {len(feature_names)} features.')

In [ ]:
# SHAP summary plot
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_vals, X_df, plot_size=None, show=False)
plt.title('SHAP Summary Plot — Churn Drivers', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print()
print('INSIGHT: Red dots = high feature value pushing prediction toward churn.')
print('Features at the top have the largest impact on predictions.')
print('Key takeaways from the beeswarm:')
print('  - High Age → higher churn probability (red dots on right)')
print('  - IsActiveMember=0 (low value) → higher churn (blue dots on right)')
print('  - NumOfProducts=1 → higher churn risk')

In [ ]:
# Waterfall plot for the highest-risk customer
y_prob_all = best_pipeline.predict_proba(X_sample)[:, 1]
high_risk_idx = np.argmax(y_prob_all)

expected_value = explainer.expected_value
if isinstance(expected_value, (list, np.ndarray)):
    expected_value = expected_value[1]

explanation = shap.Explanation(
    values=shap_vals[high_risk_idx],
    base_values=expected_value,
    data=X_df.iloc[high_risk_idx].values,
    feature_names=feature_names,
)

plt.figure(figsize=(12, 8))
shap.plots.waterfall(explanation, show=False)
plt.title(f'SHAP Waterfall — Highest Risk Customer (Prob={y_prob_all[high_risk_idx]:.3f})', fontsize=12)
plt.tight_layout()
plt.show()

## 7. Customer Risk Scores

In [ ]:
scored = pd.read_csv(os.path.join(DATA_PROCESSED, 'customer_risk_scores.csv'))
print(f'Total scored: {len(scored):,}')
print(f"Risk tier distribution:")
print(scored['Risk_Tier'].value_counts())
print()
print('Top 10 highest-risk customers:')
display_cols2 = ['Geography', 'Gender', 'Age', 'IsActiveMember', 'NumOfProducts', 
                 'Balance', 'Churn_Probability', 'Churn_Predicted', 'Risk_Tier', 'Exited']
display_cols2 = [c for c in display_cols2 if c in scored.columns]
scored.head(10)[display_cols2]